## 1. Setup & MNIST Download

Downloads the raw MNIST files, normalizes pixel values to `[0, 1]`, encodes labels as one-hot vectors, and writes two packed `f32` binary files (`data/mnist_train.bin`, `data/mnist_test.bin`) in the format expected by O.N.O (`x_size=784`, `y_size=10`, 794 floats per row).

In [1]:
import os, struct, gzip, shutil, urllib.request, subprocess, time
import numpy as np

NWORKERS = 3
NSERVERS = 2
BASE_WORKER_PORT = 50000
BASE_SERVER_PORT = 40000
WORKER_ADDRS = [f"worker-{i}:{BASE_WORKER_PORT + i}" for i in range(NWORKERS)]
SERVER_ADDRS = [f"server-{i}:{BASE_SERVER_PORT + i}" for i in range(NSERVERS)]

URLS = {
    "train_images": "https://storage.googleapis.com/cvdf-datasets/mnist/train-images-idx3-ubyte.gz",
    "train_labels": "https://storage.googleapis.com/cvdf-datasets/mnist/train-labels-idx1-ubyte.gz",
    "test_images":  "https://storage.googleapis.com/cvdf-datasets/mnist/t10k-images-idx3-ubyte.gz",
    "test_labels":  "https://storage.googleapis.com/cvdf-datasets/mnist/t10k-labels-idx1-ubyte.gz",
}

RAW_DIR = "data/mnist_raw"

TRAIN_SAMPLES_BIN = "data/mnist_train_samples.bin"
TRAIN_LABELS_BIN  = "data/mnist_train_labels.bin"
TEST_SAMPLES_BIN  = "data/mnist_test_samples.bin"
TEST_LABELS_BIN   = "data/mnist_test_labels.bin"

SAFETENSORS_PATH = "data/mnist_model.safetensors"

X_SIZE = 784
Y_SIZE = 10


def download(name, url):
    os.makedirs(RAW_DIR, exist_ok=True)
    gz = os.path.join(RAW_DIR, name + ".gz")
    raw = os.path.join(RAW_DIR, name)

    if os.path.exists(raw):
        print(f"  already exists: {raw}")
        return raw

    print(f"  downloading {name}...")
    urllib.request.urlretrieve(url, gz)
    with gzip.open(gz, "rb") as fi, open(raw, "wb") as fo:
        shutil.copyfileobj(fi, fo)
    os.remove(gz)
    return raw


def read_images(path):
    with open(path, "rb") as f:
        _, n, rows, cols = struct.unpack(">IIII", f.read(16))
        raw = f.read(n * rows * cols)

    ppi = rows * cols
    return [[b / 255.0 for b in raw[i * ppi:(i + 1) * ppi]] for i in range(n)]


def read_labels(path):
    with open(path, "rb") as f:
        _, n = struct.unpack(">II", f.read(8))
        return list(f.read(n))


def write_split_bins(images, labels, samples_out_path, labels_out_path):
    os.makedirs(os.path.dirname(samples_out_path), exist_ok=True)
    os.makedirs(os.path.dirname(labels_out_path), exist_ok=True)

    with open(samples_out_path, "wb") as fs, open(labels_out_path, "wb") as fl:
        for img, lbl in zip(images, labels):
            one_hot = [0.0] * Y_SIZE
            one_hot[lbl] = 1.0

            fs.write(struct.pack(f"{X_SIZE}f", *img))
            fl.write(struct.pack(f"{Y_SIZE}f", *one_hot))

    print(
        f"  wrote {len(images)} samples → {samples_out_path} "
        f"({os.path.getsize(samples_out_path)/1024/1024:.1f} MB)"
    )
    print(
        f"  wrote {len(labels)} labels  → {labels_out_path} "
        f"({os.path.getsize(labels_out_path)/1024/1024:.1f} MB)"
    )


for name, url in URLS.items():
    download(name, url)

if not (os.path.exists(TRAIN_SAMPLES_BIN) and os.path.exists(TRAIN_LABELS_BIN)):
    write_split_bins(
        read_images(os.path.join(RAW_DIR, "train_images")),
        read_labels(os.path.join(RAW_DIR, "train_labels")),
        TRAIN_SAMPLES_BIN,
        TRAIN_LABELS_BIN,
    )
else:
    print(f"training set already exists: {TRAIN_SAMPLES_BIN} + {TRAIN_LABELS_BIN}")

if not (os.path.exists(TEST_SAMPLES_BIN) and os.path.exists(TEST_LABELS_BIN)):
    write_split_bins(
        read_images(os.path.join(RAW_DIR, "test_images")),
        read_labels(os.path.join(RAW_DIR, "test_labels")),
        TEST_SAMPLES_BIN,
        TEST_LABELS_BIN,
    )
else:
    print(f"test set already exists: {TEST_SAMPLES_BIN} + {TEST_LABELS_BIN}")

  already exists: data/mnist_raw/train_images
  already exists: data/mnist_raw/train_labels
  already exists: data/mnist_raw/test_images
  already exists: data/mnist_raw/test_labels
training set already exists: data/mnist_train_samples.bin + data/mnist_train_labels.bin
test set already exists: data/mnist_test_samples.bin + data/mnist_test_labels.bin


## 2. Start Workers & Servers

Brings up `NSERVERS` parameter servers and `NWORKERS` workers as Docker containers. Each container logs via Docker. A short sleep is added after startup to ensure all entities are listening before the orchestrator connects.

In [2]:
import getpass

DOCKER_DIR = os.path.abspath("../../docker")
COMPOSE_FILE = os.path.abspath("../../compose.yaml")

sudo_password = getpass.getpass("sudo password: ")

# Generate compose file
subprocess.run(
    ["python3", f"{DOCKER_DIR}/gen_compose.py"],
    env={
        **os.environ,
        "WORKERS": str(NWORKERS),
        "SERVERS": str(NSERVERS),
        "RELEASE": "true",
    },
    check=True,
)

# Fill /etc/hosts with worker/server hostname translations
subprocess.run(
    ["sudo", "-S", "-E", "python3", f"{DOCKER_DIR}/fill_hosts.py"],
    input=sudo_password + "\n",
    text=True,
    env={**os.environ, "WORKERS": str(NWORKERS), "SERVERS": str(NSERVERS)},
    check=True,
)

# Bring up containers
subprocess.run(
    ["docker", "compose", "-f", COMPOSE_FILE, "up", "--build", "-d", "--remove-orphans"],
    check=True,
)

time.sleep(3)
print("All entities running.")

[sudo] contraseña para lminervino18: 

#1 [internal] load local bake definitions
#1 reading from stdin 2.21kB done
#1 DONE 0.0s

#2 [worker-2 internal] load build definition from Dockerfile
#2 transferring dockerfile: 1.03kB done
#2 DONE 0.0s

#3 [server-0 internal] load build definition from Dockerfile
#3 transferring dockerfile: 982B done
#3 DONE 0.0s

#4 [worker-2 internal] load metadata for docker.io/library/debian:bookworm-slim
#4 ...

#5 [server-1 internal] load metadata for docker.io/library/rust:1.92
#5 DONE 1.2s

#4 [worker-2 internal] load metadata for docker.io/library/debian:bookworm-slim
#4 DONE 2.0s

#6 [server-1 internal] load .dockerignore
#6 transferring context: 172B done
#6 DONE 0.0s

#7 [worker-0 chef 1/3] FROM docker.io/library/rust:1.92@sha256:f58923369ba295ae1f60bc49d03f2c955a5c93a0b7d49acfb2b2a65bebaf350d
#7 DONE 0.0s

#8 [server-0 stage-3 1/2] FROM docker.io/library/debian:bookworm-slim@sha256:f9c6a2fd2ddbc23e336b6257a5245e31f996953ef06cd13a59fa0a1df2d5c252
#8 resolve docker.io/library/debian:bookwo

 server-0  Built
 server-1  Built
 worker-1  Built
 worker-2  Built
 worker-0  Built
 Network distributed-training_training-network  Creating
 Network distributed-training_training-network  Created
 Container server-1  Creating
 Container server-0  Creating
 Container server-1  Created
 Container server-0  Created
 Container worker-1  Creating
 Container worker-0  Creating
 Container worker-2  Creating
 Container worker-1  Created
 Container worker-2  Created
 Container worker-0  Created
 Container server-0  Starting
 Container server-1  Starting
 Container server-1  Started
 Container server-0  Started
 Container worker-0  Starting
 Container worker-1  Starting
 Container worker-2  Starting
 Container worker-1  Started
 Container worker-2  Started
 Container worker-0  Started


All entities running.


## 3. Model Architecture

Defines the convolutional architecture used for both training (cell 4) and evaluation (cell 5).

Two Conv2d layers extract spatial features from the 28×28 images; a final Dense layer maps the flattened feature map to the 10 class scores. The framework automatically inserts a reshape from flat 784-dim input to `(1, 28, 28)` before the first conv, and a flatten before the Dense.

| Layer | Config | Output shape | Params |
|---|---|---|---|
| Conv2d | 8 filters, 3×3, stride=1, pad=1 | (8, 28, 28) | 80 |
| Conv2d | 16 filters, 3×3, stride=2, pad=1 | (16, 14, 14) | 1 168 |
| Dense  | → 10 | (10,) | 31 370 |

In [3]:
# Convolutional architecture for MNIST (input: flat 784-dim, reshaped internally to 1×28×28)
#
# Conv1: (1,28,28) → kernel(8,1,3) stride=1 pad=1 → (8,28,28)
# Conv2: (8,28,28) → kernel(16,8,3) stride=2 pad=1 → (16,14,14)
# Dense: 3136 → 10
#
# Flattened feature map size: 16 * 14 * 14 = 3136
CONV_LAYERS = [
    {"input_dim": (1, 28, 28), "kernel_dim": (8,  1, 3), "stride": 1, "padding": 1, "act": "sigmoid"},
    {"input_dim": (8, 28, 28), "kernel_dim": (16, 8, 3), "stride": 2, "padding": 1, "act": "sigmoid"},
]
DENSE_LAYERS = [
    {"size": 10, "act": None},
]

FLAT_AFTER_CONV = 16 * 14 * 14  # 3136

## 4. Distributed Training with O.N.O

Builds the convolutional model from `CONV_LAYERS` + `DENSE_LAYERS` with Kaiming initialization and trains it distributedly via Parameter Server (`NonBlockingSync`, `WildStore`, batch size 128, up to 50 epochs). CrossEntropy loss is used since this is a multi-class classification task. Once complete, the trained weights are saved to `data/mnist_model.safetensors`.

In [6]:
import os
import orchestra
from orchestra import Sequential, orchestrate
from orchestra.arch import Conv2d, Dense
from orchestra.activations import Sigmoid
from orchestra.initialization import Kaiming
from orchestra.datasets import LocalDataset
from orchestra.optimizers import GradientDescent
from orchestra.sync import NonBlockingSync
from orchestra.store import WildStore
from orchestra.loss_fns import CrossEntropy
from orchestra.serializer import SparseSerializer

ACT_MAP = {
    "sigmoid": Sigmoid(),
    None: None,
}

model = Sequential([
    *[
        Conv2d(
            input_dim=l["input_dim"],
            kernel_dim=l["kernel_dim"],
            stride=l["stride"],
            padding=l["padding"],
            init=Kaiming(),
            act_fn=ACT_MAP[l["act"]],
        )
        for l in CONV_LAYERS
    ],
    *[
        Dense(l["size"], Kaiming(), ACT_MAP[l["act"]])
        for l in DENSE_LAYERS
    ],
])

training = orchestra.parameter_server(
    worker_addrs=WORKER_ADDRS,
    server_addrs=SERVER_ADDRS,
    dataset=LocalDataset(TRAIN_SAMPLES_BIN, TRAIN_LABELS_BIN, x_size=784, y_size=10),
    optimizer=GradientDescent(lr=0.1),
    loss_fn=CrossEntropy(),
    sync=NonBlockingSync(),
    store=WildStore(),
    serializer=SparseSerializer(r=0.9),
    max_epochs=50,
    batch_size=128,
    offline_epochs=2,
    seed=42,
    early_stopping_tolerance=1e-6,
)

print("Starting training session...")
session = orchestrate(model, training)
trained = session.wait()
print(f"Training complete — {len(trained.weights())} parameters")

trained.save_safetensors(SAFETENSORS_PATH)
size = os.path.getsize(SAFETENSORS_PATH)
assert size > 0
print(f"Model saved to {SAFETENSORS_PATH} ({size} bytes)")

ImportError: cannot import name 'Conv2d' from 'orchestra._orchestra' (/home/lminervino18/Escritorio/Facultad/oxidized-neural-orchestra/orchestra-py/orchestra/_orchestra.cpython-312-x86_64-linux-gnu.so)

## 5. Accuracy Evaluation with PyTorch

Loads the saved `.safetensors` file into an equivalent PyTorch model. Conv2d weights are stored as `[filters, in_channels, kH, kW]` — same layout as PyTorch, so no transpose is needed. Dense weights are stored as `[in, out]` and must be transposed for PyTorch's `Linear` (`[out, in]`). The 10,000-sample MNIST test set is then evaluated and top-1 accuracy is printed.

In [ ]:
import torch
import torch.nn as nn
import numpy as np
from safetensors.torch import load_file

ACT_FN_MAP = {
    "sigmoid": torch.sigmoid,
    None: None,
}

class OnoConvNet(nn.Module):
    def __init__(self, conv_layers, dense_layers, flat_size):
        super().__init__()

        self.convs = nn.ModuleList([
            nn.Conv2d(
                in_channels=l["kernel_dim"][1],
                out_channels=l["kernel_dim"][0],
                kernel_size=l["kernel_dim"][2],
                stride=l["stride"],
                padding=l["padding"],
            )
            for l in conv_layers
        ])
        self.conv_acts = [ACT_FN_MAP[l["act"]] for l in conv_layers]

        sizes = [flat_size] + [l["size"] for l in dense_layers]
        self.linears = nn.ModuleList([
            nn.Linear(sizes[i], sizes[i + 1])
            for i in range(len(dense_layers))
        ])
        self.dense_acts = [ACT_FN_MAP[l["act"]] for l in dense_layers]

    def forward(self, x):
        # x: (batch, 784) → (batch, 1, 28, 28)
        x = x.view(-1, 1, 28, 28)
        for conv, act in zip(self.convs, self.conv_acts):
            x = conv(x)
            if act is not None:
                x = act(x)
        x = x.flatten(start_dim=1)
        for linear, act in zip(self.linears, self.dense_acts):
            x = linear(x)
            if act is not None:
                x = act(x)
        return x

state_dict = load_file(SAFETENSORS_PATH)

net = OnoConvNet(CONV_LAYERS, DENSE_LAYERS, FLAT_AFTER_CONV)

with torch.no_grad():
    # Load conv weights: ONO stores [filters, in_ch, kH, kW] — same as PyTorch, no transpose needed
    for i, conv in enumerate(net.convs):
        conv.weight.copy_(state_dict[f"layer_{i}.weight"])
        conv.bias.copy_(state_dict[f"layer_{i}.bias"])

    # Load dense weights: ONO stores [in, out] — PyTorch expects [out, in], so transpose
    conv_offset = len(net.convs)
    for j, linear in enumerate(net.linears):
        linear.weight.copy_(state_dict[f"layer_{conv_offset + j}.weight"].T)
        linear.bias.copy_(state_dict[f"layer_{conv_offset + j}.bias"])

net.eval()

x_raw = np.fromfile(TEST_SAMPLES_BIN, dtype=np.float32).reshape(-1, X_SIZE)
y_raw = np.fromfile(TEST_LABELS_BIN,  dtype=np.float32).reshape(-1, Y_SIZE)

x_test = torch.tensor(x_raw)
y_test = torch.tensor(y_raw.argmax(axis=1), dtype=torch.long)

with torch.no_grad():
    preds = net(x_test).argmax(dim=1)
    acc = (preds == y_test).float().mean().item() * 100

print(f"\nTest accuracy: {acc:.2f}%")

## 6. Stop Workers & Servers

Terminates all worker and server subprocesses started in step 2 and closes their log file handles.

In [ ]:
import subprocess

print("Stopping all entities...")
subprocess.run(
    ["docker", "compose", "-f", COMPOSE_FILE, "down"],
    check=True,
)
print("Done.")

Stopping all entities...
Done.


 Container worker-2  Stopping
 Container worker-1  Stopping
 Container worker-0  Stopping
 Container worker-2  Stopped
 Container worker-2  Removing
 Container worker-1  Stopped
 Container worker-1  Removing
 Container worker-0  Stopped
 Container worker-0  Removing
 Container worker-0  Removed
 Container worker-2  Removed
 Container worker-1  Removed
 Container server-0  Stopping
 Container server-1  Stopping
 Container server-0  Stopped
 Container server-0  Removing
 Container server-1  Stopped
 Container server-1  Removing
 Container server-1  Removed
 Container server-0  Removed
 Network distributed-training_training-network  Removing
 Network distributed-training_training-network  Removed
